<a href="https://colab.research.google.com/github/drod75/nyc_taxi_analysis/blob/main/notebooks/data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Collecting our Data

## Setup

In [1]:
# importing libraries, pandas for data manipulation, requests for data gathering, colab for environment variables
import pandas as pd
import requests
from google.colab import userdata
from google.colab import drive
from pathlib import Path
from pprint import pprint as print

In [2]:
# mounting drive for future use
drive.mount('drive')

Mounted at drive


## Data Gathering

In [3]:
# the api to query data from
data_api = "https://data.cityofnewyork.us/api/v3/views/irx2-vgm4/query.json"

In [4]:
# Headers and parameters
headers = {
    'X-App-Token': userdata.get('token'),
    "Content-Type": "application/json",
  }
payload = {
    "query":"SELECT * LIMIT 1"
}

In [5]:
# gathering temporary data
response = requests.get(data_api, headers=headers, params=payload)
data = response.json()
print(data)

[{':created_at': '2026-09-29T15:57:26.264Z',
  ':id': 'row-9zdp-t8z2.v88s',
  ':updated_at': '2026-09-29T15:57:26.264Z',
  ':version': 'rv-idqs~t3a8-8mk5',
  'airport_fee': '0.0',
  'cbd_congestion_fee': '0.0',
  'congestion_surcharge': '2.5',
  'dolocationid': '237',
  'extra': '3.5',
  'fare_amount': '10.0',
  'improvement_surcharge': '1.0',
  'mta_tax': '0.5',
  'passenger_count': '1.0',
  'payment_type': '1',
  'pulocationid': '229',
  'ratecodeid': '1.0',
  'store_and_fwd_flag': 'N',
  'tip_amount': '3.0',
  'tolls_amount': '0.0',
  'total_amount': '18.0',
  'tpep_dropoff_datetime': '2025-01-01T00:26:59.000',
  'tpep_pickup_datetime': '2025-01-01T00:18:38.000',
  'trip_distance': '1.6',
  'vendorid': '1'}]


In [6]:
# gathering the right data
query = """
SELECT
  count(*)
WHERE
  tpep_pickup_datetime >= "2025-06-21T00:00:00"
  AND
  tpep_dropoff_datetime <= "2025-09-21T23:59:59"
LIMIT 1
"""

query_payload = {
    "query": query
}

query_response = requests.get(data_api, headers=headers, params=query_payload)
query_data = query_response.json()
print(query_data)

[{'count': '11707067'}]


In [7]:
# Our goal is to collect ~11 million rows of data.

file_path = "/content/drive/MyDrive/taxi_summer_data.csv"
path = Path(file_path)
if path.exists():
    print("The path exists, skip this part!")
    taxi_df = pd.read_csv(file_path)
else:
    query = f"""
    SELECT
      count(*)
    WHERE
      tpep_pickup_datetime >= "2025-06-21T00:00:00"
      AND
      tpep_dropoff_datetime <= "2025-09-21T23:59:59"
    LIMIT 11,707,067
    OFFSET {1000*i}
    """

    query_payload = {
        "query": query
    }

    query_response = requests.get(data_api, headers=headers, params=query_payload)
    print("All Done!!!")
    query_data = query_response.json()
    taxi_df = pd.DataFrame(query_data)

'The path exists, skip this part!'


/tmp/ipykernel_1616/3369915150.py:7: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  taxi_df = pd.read_csv(file_path)


## Data Preview

In [8]:
taxi_df.head()

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,request_source
0,2,2025-06-21 00:00:00,2025-06-21 00:07:21,1.0,1.17,1.0,N,137,113,1,...,1.0,0.5,1.66,0.0,1.0,16.71,2.5,0.0,0.75,NaN
1,2,2025-06-21 00:00:00,2025-06-21 00:04:21,2.0,0.61,1.0,N,79,107,1,...,1.0,0.5,2.31,0.0,1.0,13.86,2.5,0.0,0.75,NaN
2,2,2025-06-21 00:00:00,2025-06-21 00:08:00,NaN,1.70,NaN,NaN,74,42,0,...,0.0,0.5,0.00,0.0,1.0,2.17,NaN,NaN,0.00,NaN
3,2,2025-06-21 00:00:00,2025-06-21 00:29:45,NaN,12.49,NaN,NaN,261,244,0,...,0.0,0.5,0.00,0.0,1.0,48.18,NaN,NaN,0.75,NaN
4,2,2025-06-21 00:00:00,2025-06-21 00:11:00,NaN,1.73,NaN,NaN,50,162,0,...,0.0,0.5,4.20,0.0,1.0,25.21,NaN,NaN,0.75,NaN


In [9]:
taxi_df.tail()

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,request_source
11707062,7,2025-09-21 23:58:08,2025-09-21 23:58:08,1.0,0.89,1.0,N,186,234,1,...,0.0,0.5,2.45,0.0,1.0,14.70,2.5,0.0,0.75,NaN
11707063,2,2025-09-21 23:58:15,2025-09-21 23:58:20,2.0,0.00,5.0,N,265,265,1,...,0.0,0.0,0.00,0.0,1.0,67.00,0.0,0.0,0.00,NaN
11707064,2,2025-09-21 23:58:26,2025-09-21 23:58:30,1.0,0.05,5.0,N,265,265,1,...,0.0,0.0,0.00,0.0,1.0,151.00,0.0,0.0,0.00,NaN
11707065,7,2025-09-21 23:58:54,2025-09-21 23:58:54,1.0,4.73,1.0,N,48,179,1,...,0.0,0.5,0.00,0.0,1.0,30.45,2.5,0.0,0.75,NaN
11707066,7,2025-09-21 23:59:43,2025-09-21 23:59:43,1.0,1.03,1.0,N,249,107,1,...,0.0,0.5,3.76,0.0,1.0,18.81,2.5,0.0,0.75,NaN


## Data Structure

In [10]:
print(taxi_df.shape)

(11707067, 21)


In [11]:
print(taxi_df.columns)

Index(['vendorid', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'ratecodeid', 'store_and_fwd_flag',
       'pulocationid', 'dolocationid', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'airport_fee',
       'cbd_congestion_fee', 'request_source'],
      dtype='object')


In [12]:
taxi_df.dtypes

,0
vendorid,int64
tpep_pickup_datetime,object
tpep_dropoff_datetime,object
passenger_count,float64
trip_distance,float64
ratecodeid,float64
store_and_fwd_flag,object
pulocationid,int64
dolocationid,int64
payment_type,int64


# Cleaning our Data

## Data Engineering

In [13]:
# missing values
print(taxi_df.isnull().sum())

vendorid                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count          3113406
trip_distance                  0
ratecodeid               3113406
store_and_fwd_flag       3113406
pulocationid                   0
dolocationid                   0
payment_type                   0
fare_amount                    0
extra                          0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge     3113406
airport_fee              3113406
cbd_congestion_fee             0
request_source           8744418
dtype: int64


In [14]:
# gathering all numeric columns to operate on
non_numeric_columns = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'store_and_fwd_flag', 'request_source']
numric_columns = [col for col in taxi_df.columns if col not in non_numeric_columns]
print(numric_columns)

['vendorid',
 'passenger_count',
 'trip_distance',
 'ratecodeid',
 'pulocationid',
 'dolocationid',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'airport_fee',
 'cbd_congestion_fee']


In [15]:
# numeric column adjusting
for col in numric_columns:
  taxi_df[col] = pd.to_numeric(taxi_df[col], errors='coerce')

In [18]:
# date column adjusting
taxi_df['tpep_pickup_datetime'] = pd.to_datetime(taxi_df['tpep_pickup_datetime'], format="mixed")
taxi_df['tpep_dropoff_datetime'] = pd.to_datetime(taxi_df['tpep_dropoff_datetime'], format="mixed")

In [19]:
taxi_df.head()

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,request_source
0,2,2025-06-21,2025-06-21 00:07:21,1.0,1.17,1.0,N,137,113,1,...,1.0,0.5,1.66,0.0,1.0,16.71,2.5,0.0,0.75,NaN
1,2,2025-06-21,2025-06-21 00:04:21,2.0,0.61,1.0,N,79,107,1,...,1.0,0.5,2.31,0.0,1.0,13.86,2.5,0.0,0.75,NaN
2,2,2025-06-21,2025-06-21 00:08:00,NaN,1.70,NaN,NaN,74,42,0,...,0.0,0.5,0.00,0.0,1.0,2.17,NaN,NaN,0.00,NaN
3,2,2025-06-21,2025-06-21 00:29:45,NaN,12.49,NaN,NaN,261,244,0,...,0.0,0.5,0.00,0.0,1.0,48.18,NaN,NaN,0.75,NaN
4,2,2025-06-21,2025-06-21 00:11:00,NaN,1.73,NaN,NaN,50,162,0,...,0.0,0.5,4.20,0.0,1.0,25.21,NaN,NaN,0.75,NaN


In [20]:
taxi_df.tail()

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,request_source
11707062,7,2025-09-21 23:58:08,2025-09-21 23:58:08,1.0,0.89,1.0,N,186,234,1,...,0.0,0.5,2.45,0.0,1.0,14.70,2.5,0.0,0.75,NaN
11707063,2,2025-09-21 23:58:15,2025-09-21 23:58:20,2.0,0.00,5.0,N,265,265,1,...,0.0,0.0,0.00,0.0,1.0,67.00,0.0,0.0,0.00,NaN
11707064,2,2025-09-21 23:58:26,2025-09-21 23:58:30,1.0,0.05,5.0,N,265,265,1,...,0.0,0.0,0.00,0.0,1.0,151.00,0.0,0.0,0.00,NaN
11707065,7,2025-09-21 23:58:54,2025-09-21 23:58:54,1.0,4.73,1.0,N,48,179,1,...,0.0,0.5,0.00,0.0,1.0,30.45,2.5,0.0,0.75,NaN
11707066,7,2025-09-21 23:59:43,2025-09-21 23:59:43,1.0,1.03,1.0,N,249,107,1,...,0.0,0.5,3.76,0.0,1.0,18.81,2.5,0.0,0.75,NaN


## Missing Values

In [21]:
# missing values
print(taxi_df.isnull().sum())

vendorid                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count          3113406
trip_distance                  0
ratecodeid               3113406
store_and_fwd_flag       3113406
pulocationid                   0
dolocationid                   0
payment_type                   0
fare_amount                    0
extra                          0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge     3113406
airport_fee              3113406
cbd_congestion_fee             0
request_source           8744418
dtype: int64


In [22]:
# passenger_count value counts
print(taxi_df['passenger_count'].value_counts(dropna=False))

passenger_count
1.0    6715887
NaN    3113406
2.0    1226103
3.0     308379
4.0     225030
0.0      61180
5.0      36196
6.0      20849
8.0         16
9.0         15
7.0          6
Name: count, dtype: int64


In [23]:
# ratecode value counts
"""
Possible values

1= Standard rate
2=JFK
3=Newark
4=Nassau or Westchester
5=Negotiated fare
6=Group ride
99 = Null/unknown

"""
print(taxi_df['ratecodeid'].value_counts(dropna=False))

ratecodeid
1.0     8059782
NaN     3113406
2.0      329229
5.0      126872
3.0       41044
4.0       34420
99.0       2306
6.0           8
Name: count, dtype: int64


In [24]:
# store_and_fwd_flag value counts
"""
Possible Values

Y= store and forward trip
N= not a store and forward trip

"""
print(taxi_df['store_and_fwd_flag'].value_counts(dropna=False))

store_and_fwd_flag
N      8568175
NaN    3113406
Y        25486
Name: count, dtype: int64


In [25]:
# congestion_surcharge value counts
print(taxi_df['congestion_surcharge'].value_counts(dropna=False))

congestion_surcharge
 2.50    7687339
 NaN     3113406
 0.00     727295
-2.50     179021
 1.00          5
 0.75          1
Name: count, dtype: int64


In [26]:
# airport_fee value counts
print(taxi_df['airport_fee'].value_counts(dropna=False))

airport_fee
 0.00    7700775
 NaN     3113406
 1.75     834200
-1.75      52905
 6.75       4535
 5.00       1245
 0.50          1
Name: count, dtype: int64


In [27]:
# request_source value counts
"""
Possible Values

Origin of e-hail request. ID's can be provided for forwarded trips:

"CC": Call center (required for accessible dispatch),

"A": Company's own app,

"HVXXXX": High volume ID,

"EHXXXX": Ehail ID

"""
print(taxi_df['request_source'].value_counts(dropna=False))

request_source
NaN       8744418
HV0003    2703632
A          182336
EH0010      53209
EH0004      23255
CC            217
Name: count, dtype: int64


## Exporting

In [ ]:
# if path.exists():
#     print("The path exists, skip this part!")
# else:
#     taxi_df.to_csv(file_path, index=False)